# UNO Vision — IAPR 2026 Final Report

**Task** — for each UNO game image (4000×2662 px), predict the center card, the active player (token), and each of the 4 players' hand cards.

**Metric** — `Score = 0.1·CenterAcc + 0.1·ActiveAcc + 0.8·F1` (multi-set F1).

**Constraints** — ≤ 12M parameters total, **no pre-trained model**, **no external dataset**, trained *from scratch*.

**Baselines** — classical 0.570 · Deep Learning 0.647.

---

## Final result: **Score 0.850** — CenterAcc 0.938 · ActiveAcc 0.802 · F1 0.845 — 11.08M-param pipeline.

This report tells the **chronological story of three versions**. The central thread: *V3 (corner detection) fails as a pipeline, but its corner detector — a by-product of a "failed" exploration — unlocks the two final gains that take V2 from 0.787 to 0.850.*

| Version | Guiding idea | Score |
|---|---|---|
| **V1** Hybrid baseline | classical HSV detection + from-scratch CNN + auto-labeling | **0.764** |
| **V2** Industrialization | synthetic data + learned detector + distillation | **0.787** |
| **V3** Corner detection | detect / classify the card *corners* | 0.41 (direct) |
| **Convergence (final V2)** | V3 detector → iterative auto-label + 0-param center ensemble | **0.850** |

In [ ]:
import sys, warnings
from pathlib import Path
from collections import Counter
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import TRAIN_CSV, TRAIN_DIR, DATA_DIR, MODELS_DIR
TEMPLATES_DIR = DATA_DIR / 'card_templates'
print('Repo root:', ROOT)

## 1. The dataset

81 annotated images. `train.csv` gives, per image: center card, active player, and the 4 players' hands. 626 manually drawn bounding boxes (single "card" class). 54 card classes. Two background types: white and noisy tropical.

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print(f'{df.shape[0]} annotated images')
df.head(4)

In [ ]:
cards = []
for col in ['player_1_cards','player_2_cards','player_3_cards','player_4_cards','center_card']:
    for v in df[col].dropna():
        cards += [c for c in str(v).split(';') if c and c != 'EMPTY']
cc = Counter(cards); ks = sorted(cc)
fig, ax = plt.subplots(figsize=(15,3))
ax.bar(ks,[cc[k] for k in ks]); ax.tick_params(axis='x',rotation=90,labelsize=6)
ax.set_title(f'Distribution of {sum(cc.values())} cards ({len(cc)} classes)')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(18,6))
for ax, iid in zip(axes, ['L1000770','L1000909']):
    p = TRAIN_DIR / f'{iid}.jpg'
    if p.exists():
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        r = df[df.image_id==iid].iloc[0]
        ax.set_title(f'{iid} | center={r.center_card} active={r.active_player}')
    ax.axis('off')
fig.suptitle('White background (left) vs noisy tropical background (right)')
plt.tight_layout(); plt.show()

## 2. Version 1 — Hybrid classical + Deep Learning baseline

Four-stage pipeline, **zero pre-training**:
1. **Card detection (classical, 0 params)**: blurred HSV saturation (white bg) + *enclosed* white ovals (noisy bg); per-color split for stacks.
2. **UnoCNN classifier** (ResNet-18-light) trained *from scratch* on the 54 templates + augmentation.
3. **Token** (HSV, adaptive V threshold from the median).
4. **Geometric assignment** (distance to image center + angle).

### Key technique: auto-labeling (+0.27)
A classifier trained on templates only plateaus at **0.42** (domain gap: perfect templates vs worn/blurry real cards). Auto-labeling real cards (color+zone match against GT) → 389 real crops → fine-tune → **0.42 → 0.69**. *This "better data" lever becomes the project's technical backbone.*

In [ ]:
tfiles = sorted(TEMPLATES_DIR.glob('*.png'))
print(f'{len(tfiles)} templates 200x300 extracted from reference_images')
fig, axes = plt.subplots(2,9,figsize=(18,5))
for ax,f in zip(axes.flat, tfiles[:18]):
    ax.imshow(cv2.cvtColor(cv2.imread(str(f)),cv2.COLOR_BGR2RGB)); ax.set_title(f.stem,fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from src.augmentation import augment, AugConfig
tpl = cv2.imread(str(TEMPLATES_DIR/'r_5.png')); rng = np.random.default_rng(0)
fig, axes = plt.subplots(1,8,figsize=(18,3))
axes[0].imshow(cv2.cvtColor(tpl,cv2.COLOR_BGR2RGB)); axes[0].set_title('original'); axes[0].axis('off')
for ax in axes[1:]:
    ax.imshow(cv2.cvtColor(augment(tpl, rng, AugConfig()),cv2.COLOR_BGR2RGB)); ax.axis('off')
fig.suptitle('From-scratch augmentations (rotation, perspective, noise, corner-preserving partial crop)')
plt.tight_layout(); plt.show()

## 3. Version 2 — Industrialization

### 3.1 Quantified failure analysis (mandatory first step)
Decomposing the baseline's 182 card errors into 6 categories:

| Category | % of card errors |
|---|---|
| **Misclassification** (card localized, wrong label) | **58.8 %** |
| FP_detection (hallucination) | 28.6 % |
| FN_detection (missed card) | 9.9 % |
| wrong_player (right label, wrong zone) | 2.7 % |

→ 88 % is detection+classification; player assignment is already near-perfect (**do not touch it**). All effort goes to detection/classification quality.

### 3.2 V2 building blocks
- **5000 synthetic scenes** composed from internal assets only (54 templates + backgrounds extracted from the 81 real images outside bboxes) → exact annotations by construction.
- **Learned detector** CenterNet (objectness heatmap + bbox regression), channels scaled [16,32,64,96]→**[40,80,160,320] (4.77M)**, 2-phase training (synth 80% + real 20%, then real fine-tune). *Bug fixed*: focal-loss `pos_mask = (target == 1.0)` never selected a pixel (discrete Gaussian max < 1.0) → force `target = 1.0` at the center pixel.
- **Distillation**: teacher 11.2M → **student 6.31M** (loss `0.3·CE + 0.7·KL·T²`, T=4), val_acc 0.925, to fit the parameter budget.
- 0-param gains: token *double-try fallback* (ActiveAcc 0.716→0.790), confidence threshold 0.50, center card exempt from confidence filter.

**V2 = 0.787**, budget **4.77M + 6.31M = 11.08M < 12M**.

In [ ]:
# Parameter count of the FINAL pipeline (detector 4.77M + student 6.31M)
from src.detector import CardDetector
from src.model import UnoCNN
dck = torch.load(MODELS_DIR/'detector.pt', map_location='cpu', weights_only=False)
cck = torch.load(MODELS_DIR/'classifier.pt', map_location='cpu', weights_only=False)
dch = tuple(dck.get('channels',(40,80,160,320))); cch = tuple(cck.get('channels',(48,96,192,384)))
det = CardDetector(channels=dch); clf = UnoCNN(num_classes=54, channels=cch)
nd = sum(p.numel() for p in det.parameters()); ncf = sum(p.numel() for p in clf.parameters())
print(f'CenterNet detector  {dch} : {nd/1e6:.2f}M')
print(f'Student classifier  {cch} : {ncf/1e6:.2f}M')
print(f'TOTAL : {(nd+ncf)/1e6:.2f}M / 12M  ->  {"OK" if nd+ncf < 12e6 else "OVER BUDGET"}')

## 4. Version 3 — The "corner detection" approach

**Hypothesis**: UNO prints color+value in two diagonally opposite corners. Detecting/classifying these corners should solve occlusion (one corner is enough) and attack the 59 % misclassification (a clean corner crop, free of the central oval's clutter).

8-phase process with check-points. Per-component results:

| Component | Metric | Value |
|---|---|---|
| Corner detector (3.19M) | real F1 | **0.985** (P=0.98 R=0.99) |
| Corner classifier (1.30M) | train / **real** acc | 0.999 / **0.759** |

**Direct failure**: 6 variants tested, best **0.741 < V2 0.787**. Three root causes:
1. **Synthetic→real gap** of the corner classifier (0.999 → 0.759): synthetic corners do not reproduce wear/reflections/blur.
2. **Error multiplication**: `det 0.99 × clf/corner 0.76 × 2-corner agreement` ≪ a full-card classifier (full context in one pass).
3. **Pairing dilemma**: label-agreement → 1-corner cards mislocated (CenterAcc 0.07); pure geometric → neighbor mis-pairing (F1 0.38).

> **Lesson**: an excellent sub-component (detector 0.99) does not save a pipeline when uncertainty composition and a domain gap dominate. **But the R=0.99 corner detector is a reusable asset** — it is what unlocks the rest.

## 5. The convergence — how V3 unlocks V2 (0.787 → 0.850)

### 5.1 Iterative auto-labeling via the corner detector (0.787 → 0.820)
The V1 pattern replayed ("better detector → better data → better classifier"): the v3 corner detector (R=0.99, occlusion-robust) localizes the cards in the 81 images → **406 clean real crops, 54/54 classes** (vs 237/53 in V1, +71 %, the `wild` class finally covered). Fine-tuning the student on them → **F1 0.797 → 0.844**, Score 0.787 → **0.820**.

### 5.2 0-param center-card ensemble (0.820 → 0.850)
In hybrid mode, **two detectors already run** (heuristic for the center, learned 4.77M for players); we only used the heuristic view of the center. **0-parameter ensemble**: keep *both* center views, classify both, pick the **most confident** among candidates near the image center. → **CenterAcc 0.642 → 0.938** (> baseline 11.2M's 0.901!), Score → **0.850**.

*Variants tested and rejected (rigor)*: combined crops 237+406 (0.814, dilution); NCC template matching for the center (0.21, not rotation-invariant).

## 6. Final pipeline visualization (detection + classification)

In [ ]:
from src.inference import Classifier, predict_scene
from src.detector_inference import CardDetectorRuntime
classifier = Classifier(MODELS_DIR/'classifier.pt')
detector = CardDetectorRuntime(MODELS_DIR/'detector.pt', device=str(classifier.device))
print('Final pipeline: hybrid (heuristic center + learned detector players + 0-param center ensemble)')

In [ ]:
def show_pred(iid):
    img = cv2.imread(str(TRAIN_DIR/f'{iid}.jpg'))
    pr = predict_scene(img, iid, classifier, use_tta=True, detector=detector, hybrid=True)
    r = df[df.image_id==iid].iloc[0]
    plt.figure(figsize=(13,8)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'{iid}\nGT   center={r.center_card}  active={r.active_player}\n'
              f'PRED center={pr.center_card}  active={pr.active_player}', fontsize=11)
    plt.show()
    print('GT   p1..p4 :', [r.player_1_cards,r.player_2_cards,r.player_3_cards,r.player_4_cards])
    print('PRED p1..p4 :', [';'.join(pr.player_cards[p]) or 'EMPTY' for p in ('p1','p2','p3','p4')])
show_pred('L1000770')

In [ ]:
show_pred('L1000909')  # noisy tropical background

## 7. Quantitative evaluation on the 81 images (final pipeline)

Final pipeline = detector 4.77M + student 6.31M + 0-param center ensemble, 8× TTA enabled. (~4 min on GPU.)

In [ ]:
def parse_hand(s): return [] if (not s or s=='EMPTY') else str(s).split(';')
def f1m(p,g):
    pc,gc=Counter(p),Counter(g); tp=sum((pc&gc).values())
    fp=sum(pc.values())-tp; fn=sum(gc.values())-tp
    return 1.0 if (2*tp+fp+fn)==0 else 2*tp/(2*tp+fp+fn)
rows=[]
for r in df.itertuples(index=False):
    pth=TRAIN_DIR/f'{r.image_id}.jpg'
    if not pth.exists(): continue
    pr=predict_scene(cv2.imread(str(pth)), r.image_id, classifier, use_tta=True, detector=detector, hybrid=True)
    gt=[]; pd_=[]
    for i,s in enumerate(['p1','p2','p3','p4']):
        gt+=parse_hand(getattr(r,f'player_{i+1}_cards')); pd_+=pr.player_cards[s]
    rows.append({'image_id':r.image_id,'center_ok':int(pr.center_card==r.center_card),
                 'active_ok':int(pr.active_player==r.active_player),'f1':f1m(pd_,gt)})
res=pd.DataFrame(rows)
ca,aa,f1=res.center_ok.mean(),res.active_ok.mean(),res.f1.mean()
score=0.1*ca+0.1*aa+0.8*f1
print(f'CenterAcc {ca:.3f} | ActiveAcc {aa:.3f} | F1 {f1:.3f}')
print(f'SCORE = {score:.3f}   (classical baseline 0.570 - DL 0.647)')

## 8. Ablation — the chronological progression of the 3 versions

In [ ]:
abl = {'V1 baseline\nhybrid':0.764,'V2 synth+\ndistill':0.787,'V3 corner\n(direct)':0.741,
       'V2 +iterative\nauto-label (V3)':0.820,'V2 final +center\nensemble':0.850}
fig,ax=plt.subplots(figsize=(11,4))
bars=ax.bar(list(abl),list(abl.values()),color=['#888','#3498db','#e67e22','#16a085','#2ecc71'])
ax.axhline(0.647,ls='--',c='r',label='DL baseline 0.647'); ax.set_ylim(0.6,0.9)
ax.set_ylabel('Score'); ax.legend()
for b,v in zip(bars,abl.values()): ax.text(b.get_x()+b.get_width()/2,v+0.005,f'{v:.3f}',ha='center')
ax.set_title('V3 fails directly (0.741) but unlocks V2 -> 0.850')
plt.tight_layout(); plt.show()

| Technique | Δ Score |
|---|---|
| V1 auto-labeling (templates → +389 real crops) | +0.27 (F1) |
| Learned detector + synthetic + distillation | 0.764 → 0.787 |
| Iterative auto-labeling via v3 corner detector | 0.787 → 0.820 |
| 0-param center ensemble (2 detector views) | 0.820 → **0.850** |

**Rejected (rigor)**: OBB+FPN (collapsed bbox, saturated sigmoid); pure V3 corner (synthetic gap); combined crops (dilution); center NCC template (not rotation-invariant).

## 9. Failure analysis (remaining cases)

In [ ]:
print('5 worst images (F1):')
print(res.nsmallest(5,'f1')[['image_id','f1','center_ok','active_ok']].to_string(index=False))
print(f"\nCenter wrong: {(res.center_ok==0).sum()}/{len(res)} | "
      f"Active wrong: {(res.active_ok==0).sum()}/{len(res)}")
print('Residual errors: heavily occluded cards (1 edge visible) + near digit confusions (b_5/b_2) + yellow token on foliage.')
fig,axes=plt.subplots(1,3,figsize=(20,6))
for ax,iid in zip(axes, res.nsmallest(3,'f1').image_id):
    ax.imshow(cv2.cvtColor(cv2.imread(str(TRAIN_DIR/f'{iid}.jpg')),cv2.COLOR_BGR2RGB))
    rr=res[res.image_id==iid].iloc[0]; ax.set_title(f'{iid}  F1={rr.f1:.2f}'); ax.axis('off')
fig.suptitle('Hardest cases'); plt.tight_layout(); plt.show()

## 10. Conclusion

An iterative engineering process driven by **quantified failure analysis** rather than intuition:

1. **V1** proves the hybrid classical+DL design and reveals the data lever (auto-labeling, +0.27).
2. **V2** industrializes it (legal synthetic data, learned detector, distillation to fit the 12M budget) → 0.787; two bugs/failures documented (focal-loss, OBB).
3. **V3** explores corner detection: it *fails as a pipeline* (synthetic gap, error multiplication) **but produces the corner detector** which, re-injected into V2 via iterative auto-labeling (→ 0.820) then a 0-param center ensemble (→ 0.850), delivers the final result.

**Final score 0.850** (+0.086 vs classical baseline, +0.203 vs DL), 11.08M-param pipeline, *from scratch*, internal data only.

> *Transferable lesson: a component built for an abandoned approach can be the decisive ingredient of another — the "failed" exploration has instrumental value.*

### Reproducibility
```bash
pip install -r requirements.txt
python scripts/download_data.py
python scripts/extract_templates.py
# final pipeline provided: outputs/models/{detector.pt 4.77M, classifier.pt student 6.31M}
python main.py --tta --hybrid   # -> outputs/submissions/submission.csv (Score 0.850)
```
Full 3-version write-up: `reports/RAPPORT_FINAL.md`. V3 study: `reports/v3_*`.